In [ ]:
# ==============================================================
# 04 – DRL Training (Credit Decision Environment + DQN)
# Foundation for Single-Agent PPO and Multi-Agent MARL
# Supports RQ1 (DRL vs traditional) and RQ4 (business impact)
# ==============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from collections import deque, namedtuple
import random
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

ROOT = Path(".")
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_SYNTHETIC = ROOT / "data" / "synthetic"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1. Load Features (prefer fused / CNN embeddings if available)
# --------------------------------------------------------------
# Try to load CNN embeddings first (from notebook 03)
emb_path = DATA_PROCESSED / "cnn_embeddings.npy"
label_path = DATA_PROCESSED / "cnn_labels.npy"
thin_path = DATA_PROCESSED / "cnn_thin.npy"

if emb_path.exists():
    X = np.load(emb_path)
    y = np.load(label_path)
    thin = np.load(thin_path)
    print(f"Loaded CNN embeddings: {X.shape}")
else:
    # Fallback: create simple features from global dataset
    df = pd.read_csv(DATA_SYNTHETIC / "global_credit_from_german.csv")
    feature_cols = ['duration', 'credit_amount', 'age', 'thin_file']
    feature_cols = [c for c in feature_cols if c in df.columns]
    X = df[feature_cols].values.astype(np.float32)
    y = df['default'].values.astype(np.int64)
    thin = df['thin_file'].values.astype(np.int64)
    print(f"Loaded tabular features: {X.shape}")

print(f"Default rate: {y.mean():.2%} | Thin-file rate: {thin.mean():.2%}")

# --------------------------------------------------------------
# 2. Credit Decision Environment (Gym-style)
# --------------------------------------------------------------
class CreditDecisionEnv:
    """
    Custom Environment for Credit Decisioning.
    Actions: 0 = Reject, 1 = Approve, 2 = Counter-Offer (approve with lower amount)
    """
    def __init__(self, X, y, thin, cost_fn=5.0, cost_fp=1.0, reward_approve_good=1.0):
        self.X = X
        self.y = y                  # 1 = default (bad), 0 = good
        self.thin = thin
        self.cost_fn = cost_fn      # cost of False Negative (approve bad customer)
        self.cost_fp = cost_fp      # cost of False Positive (reject good customer)
        self.reward_approve_good = reward_approve_good
        self.n_samples = len(X)
        self.action_space = 3
        self.observation_space = X.shape[1]
        self.current_idx = 0
        self.reset()

    def reset(self):
        self.indices = np.random.permutation(self.n_samples)
        self.current_idx = 0
        self.total_reward = 0
        return self._get_state()

    def _get_state(self):
        idx = self.indices[self.current_idx]
        return self.X[idx], idx

    def step(self, action):
        idx = self.indices[self.current_idx]
        true_label = self.y[idx]          # 1 = bad, 0 = good
        is_thin = self.thin[idx]

        # Reward design (business aligned)
        if action == 1:  # Approve
            if true_label == 0:  # Good customer
                reward = self.reward_approve_good
                if is_thin:
                    reward += 0.3   # bonus for including thin-file good customers (RQ4)
            else:  # Bad customer
                reward = -self.cost_fn
        elif action == 2:  # Counter-offer (partial approve)
            if true_label == 0:
                reward = self.reward_approve_good * 0.6
            else:
                reward = -self.cost_fn * 0.5
        else:  # Reject
            if true_label == 1:  # Correctly rejected bad
                reward = 0.4
            else:  # Rejected good customer
                reward = -self.cost_fp
                if is_thin:
                    reward -= 0.2   # extra penalty for excluding thin-file good customers

        self.total_reward += reward
        self.current_idx += 1
        done = self.current_idx >= self.n_samples

        if not done:
            next_state, _ = self._get_state()
        else:
            next_state = np.zeros(self.observation_space, dtype=np.float32)

        return next_state, reward, done, {"true_label": true_label, "is_thin": is_thin}

# --------------------------------------------------------------
# 3. DQN Agent
# --------------------------------------------------------------
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim=3, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim)
        )
    def forward(self, x):
        return self.net(x)

class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = zip(*batch)
        return (np.array(state), action, reward, np.array(next_state), done)
    def __len__(self):
        return len(self.buffer)

# --------------------------------------------------------------
# 4. Training Loop
# --------------------------------------------------------------
state_dim = X.shape[1]
action_dim = 3

policy_net = DQN(state_dim, action_dim).to(device)
target_net = DQN(state_dim, action_dim).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
buffer = ReplayBuffer()

env = CreditDecisionEnv(X, y, thin)

# Hyperparameters
BATCH_SIZE = 64
GAMMA = 0.95
EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY = 0.995
TARGET_UPDATE = 10
NUM_EPISODES = 80

epsilon = EPS_START
episode_rewards = []

print("\nStarting DQN Training...")
for episode in range(1, NUM_EPISODES + 1):
    state, _ = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device)
    total_reward = 0
    done = False

    while not done:
        # Epsilon-greedy
        if random.random() < epsilon:
            action = random.randrange(action_dim)
        else:
            with torch.no_grad():
                q_values = policy_net(state.unsqueeze(0))
                action = q_values.argmax(dim=1).item()

        next_state, reward, done, info = env.step(action)
        total_reward += reward

        buffer.push(state.cpu().numpy(), action, reward, next_state, done)
        state = torch.tensor(next_state, dtype=torch.float32, device=device)

        # Learn
        if len(buffer) >= BATCH_SIZE:
            states, actions, rewards, next_states, dones = buffer.sample(BATCH_SIZE)
            states = torch.tensor(states, dtype=torch.float32, device=device)
            actions = torch.tensor(actions, dtype=torch.long, device=device)
            rewards = torch.tensor(rewards, dtype=torch.float32, device=device)
            next_states = torch.tensor(next_states, dtype=torch.float32, device=device)
            dones = torch.tensor(dones, dtype=torch.float32, device=device)

            q_values = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze()
            next_q = target_net(next_states).max(1)[0].detach()
            target = rewards + GAMMA * next_q * (1 - dones)

            loss = F.mse_loss(q_values, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    epsilon = max(EPS_END, epsilon * EPS_DECAY)
    episode_rewards.append(total_reward)

    if episode % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if episode % 10 == 0 or episode == 1:
        avg_reward = np.mean(episode_rewards[-10:])
        print(f"Episode {episode:3d}/{NUM_EPISODES} | Avg Reward (last 10): {avg_reward:7.2f} | Epsilon: {epsilon:.3f}")

# --------------------------------------------------------------
# 5. Save Model & Results
# --------------------------------------------------------------
torch.save(policy_net.state_dict(), RESULTS / "dqn_credit_agent.pt")
np.save(RESULTS / "dqn_episode_rewards.npy", np.array(episode_rewards))

print(f"\n✓ DQN model saved to results/dqn_credit_agent.pt")

# Plot learning curve
plt.figure(figsize=(10, 5))
plt.plot(episode_rewards, alpha=0.6, label="Episode Reward")
plt.plot(pd.Series(episode_rewards).rolling(10).mean(), label="Moving Average (10)", color="red")
plt.title("DQN Training – Credit Decision Environment")
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.legend()
plt.grid(True)
plt.savefig(RESULTS / "dqn_learning_curve.png", dpi=140, bbox_inches="tight")
plt.show()

print("\n✅ 04_DRL_Training completed.")
print("Next → 05_Feature_Fusion.ipynb or 06_Single_Agent_PPO.ipynb")